In [35]:
from google.cloud import bigquery

client = bigquery.Client()

query = """
    WITH user_ingredients AS (
    SELECT DISTINCT LOWER(TRIM(ing)) AS ing
    FROM UNNEST(['onion', 'butter', 'apple', 'pepper', 'tomato']) AS ing
)

SELECT*
FROM `wagon-bootcamp-501612-i1.recipes_clean_300.recipes_final_array` r

WHERE ARRAY_LENGTH(r.ingredients_clean) > 0
    AND NOT EXISTS (
    SELECT 1
    FROM UNNEST(r.ingredients_clean) AS recipe_ing
    WHERE LOWER(TRIM(recipe_ing)) NOT IN (
    SELECT ing
    FROM user_ingredients
    )
);
"""

job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter(
                "user_ing_list", "STRING", ['chocolate', 'butter', 'chocolate', 'lemon juice']
            )
        ]
    )

df = client.query(query, job_config=job_config).to_dataframe()

# df = client.query(query).to_dataframe()
df.head()

/home/samuel/.pyenv/versions/master_shelf/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,name,ingredients,ingredients_raw,steps,servings,persons,portion_size,ingredients_clean,type_dish,type_diet,type_meal,type_occasion,type_origin,time_to_make


In [91]:

test = df.iloc[250]


In [92]:
df = df.sample(250)

In [179]:
df_clean = df.drop(columns=['name','ingredients_raw', 'steps', 'servings', 'persons','portion_size','ingredients_clean'])

In [94]:
df_clean.head()

,type_dish,type_diet,type_meal,type_occasion,type_origin,time_to_make
43819,[],"[dietary, low-carb, low-calorie]",[],[],[],60
57775,[],"[dietary, high-protein, low-carb, low-sodium, ...","[main-dish, dinner-party]",[dinner-party],[],60
65141,"[salads, vegetables]","[dietary, vegetarian]",[dinner-party],"[dinner-party, summer]",[north-american],15
5890,[meatballs],"[dietary, gluten-free]","[main-dish, appetizers]",[],[],60
65496,[],"[dietary, low-carb, very-low-carbs]","[dinner-party, appetizers]","[dinner-party, new-years]",[],15


In [95]:
df_clean.head()

,type_dish,type_diet,type_meal,type_occasion,type_origin,time_to_make
43819,[],"[dietary, low-carb, low-calorie]",[],[],[],60
57775,[],"[dietary, high-protein, low-carb, low-sodium, ...","[main-dish, dinner-party]",[dinner-party],[],60
65141,"[salads, vegetables]","[dietary, vegetarian]",[dinner-party],"[dinner-party, summer]",[north-american],15
5890,[meatballs],"[dietary, gluten-free]","[main-dish, appetizers]",[],[],60
65496,[],"[dietary, low-carb, very-low-carbs]","[dinner-party, appetizers]","[dinner-party, new-years]",[],15


In [214]:
user =  {'time_max': '15',
 'occasion': ['summer'],
 'dish': [],
 'meal': ['main-dish'],
 'diet': ['gluten-free', 'dietary'],
 'origin': [],
 'pantry_items': ['chocolate', 'butter', 'chocolate', 'lemon juice']}

In [228]:
user2 =  {'time_max': "0",
 'occasion': [],
 'dish': [],
 'meal': [],
 'diet': [],
 'origin': [],
 'pantry_items': []}

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
from numpy import hstack
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import ParameterGrid


def predict_closer(user , df_clean):
    #On créé les mlb pour chaque colonnes à encoder (permettra de faire des 'groupes de colonnes avec le même poids, tout en choisissant les poids')
    mlb_dish = MultiLabelBinarizer()
    mlb_diet = MultiLabelBinarizer()
    mlb_meal = MultiLabelBinarizer()
    mlb_occasion = MultiLabelBinarizer()
    mlb_origin = MultiLabelBinarizer()
    mlb_time_to_make = MultiLabelBinarizer()


    param_grid = {'dish':[3], 'diet':[5], 'meal':[1], 'occasion':[1],
              'origin':[1], 'time_to_make':[3]}

    grid = list(ParameterGrid(param_grid))


    # On applique les mlb aux colonnes
    X_dish = mlb_dish.fit_transform(df_clean['type_dish'])
    X_diet = mlb_diet.fit_transform(df_clean['type_diet'])
    X_meal = mlb_meal.fit_transform(df_clean['type_meal'])
    X_occasion = mlb_occasion.fit_transform(df_clean['type_occasion'])
    X_origin = mlb_origin.fit_transform(df_clean['type_origin'])
    X_time_to_make = mlb_time_to_make.fit_transform(df_clean['time_to_make'])

    history = get_history()

    if not user["dish"] and not user["origin"] and user["time_max"] == "0" and not user["diet"] and not user["meal"] and not user["occasion"]:
        X_test_dish = mlb_dish.transform([[max(set(history["dish"]), key=history["dish"].count)]])
        X_test_diet = mlb_diet.transform([[max(set(history["diet"]), key=history["diet"].count)]])
        X_test_meal = mlb_meal.transform([[max(set(history["meal"]), key=history["meal"].count)]])
        X_test_occasion = mlb_occasion.transform([[max(set(history["occasion"]), key=history["occasion"].count)]])
        X_test_origin = mlb_origin.transform([[max(set(history["origin"]), key=history["origin"].count)]])
        X_test_time_to_make = mlb_time_to_make.transform([[max(set(history["time_max"]), key=history["time_max"].count)]])
    else:
        X_test_dish = mlb_dish.transform([user["dish"]])
        X_test_diet = mlb_diet.transform([user['diet']])
        X_test_meal = mlb_meal.transform([user["meal"]])
        X_test_occasion = mlb_occasion.transform([user['occasion']])
        X_test_origin = mlb_origin.transform([user['origin']])
        X_test_time_to_make = mlb_time_to_make.transform([user['time_max']])


    result = []
    for param in grid:
        X = hstack([X_dish * param['dish'],X_diet * param['diet'],X_meal*param['meal'],
                X_occasion * param['occasion'],X_origin * param['origin'],
                X_time_to_make * param['time_to_make']])


        X_test = hstack([X_test_dish * param['dish'], X_test_diet * param['diet'],X_test_meal*param['meal'],
                        X_test_occasion * param['occasion'],X_test_origin * param['origin'],
                        X_test_time_to_make * param['time_to_make']])

        model = NearestNeighbors(
            n_neighbors=5,
            metric="cosine")

        model.fit(X)
        distances, indices = model.kneighbors(X_test)
        result.append({
        **param,
        "score": distances,
        "recipe_index": indices
    })
        print(X_test)
    return result


In [253]:
result = predict_closer(user2 , df_clean)
result

/home/mitri/.pyenv/versions/master_shelf/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['15'] will be ignored
  warnings.warn(


[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 3 0 0
  0 0 5 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0]]


[{'diet': 5,
  'dish': 3,
  'meal': 1,
  'occasion': 1,
  'origin': 1,
  'time_to_make': 3,
  'score': array([[0.19461273, 0.20196802, 0.20196802, 0.20196802, 0.20196802]]),
  'recipe_index': array([[54817, 53658, 32839, 66356, 36608]])}]

In [248]:
for i in list(result[0]["recipe_index"][0]):
    print(df_clean.iloc[i])

type_dish        [vegetables]
type_diet           [dietary]
type_meal         [main-dish]
type_occasion              []
type_origin        [european]
time_to_make               30
Name: 54817, dtype: object
type_dish             [vegetables]
type_diet                [dietary]
type_meal              [main-dish]
type_occasion                   []
type_origin      [european, greek]
time_to_make                    30
Name: 53658, dtype: object
type_dish             [vegetables]
type_diet                [dietary]
type_meal              [main-dish]
type_occasion                   []
type_origin      [european, swiss]
time_to_make                    60
Name: 32839, dtype: object
type_dish               [vegetables]
type_diet                  [dietary]
type_meal                [main-dish]
type_occasion                     []
type_origin      [european, italian]
time_to_make                      30
Name: 66356, dtype: object
type_dish               [vegetables]
type_diet                  [dieta

In [223]:
if not user["dish"] and not user["origin"] and not user["time_max"] and not user["diet"] and not user["meal"] and not user["occasion"]:
    print("y")
else:
    print("n")

n


In [236]:
[max(set(history["dish"]), key=history["dish"].count)]

['vegetables']

In [245]:
[user["dish"]]

[[]]

In [251]:
def get_history():
    history = {'time_max': ['15',"30", "15", "15"],
    'occasion': ['summer', "spring", "brunch", "spring"],
    'dish': ["vegetables","pasta" , "vegetables" , "vegetables"],
    'meal': ['main-dish', 'main-dish', 'main-dish', 'main-dish', 'dessert'],
    'diet': ['gluten-free', 'dietary', 'dietary', 'dietary' , 'dietary', 'gluten-free'],
    'origin': ['american', "european", "european", "european" , "american"]}
    return history